# Treino final — LightGBM (produção)

Notebook de **treino e exportação** do modelo vencedor definido em `scripts/abt_to_model_home_credit_test.ipynb`.

- Configuração central: `config/model_config.yaml`
- Split: **80% treino** | **19,9% teste** | **0,1% demo** (holdout cego para avaliação individual)
- Modelo: LightGBM | ABT full | threshold de negócio **0,25**


In [ ]:
# Opcional em ambiente novo
# !pip install lightgbm scikit-learn pyarrow pyyaml pandas matplotlib seaborn s3fs


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'config' / 'model_config.yaml').exists() and (ROOT.parent / 'config' / 'model_config.yaml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.model_config import get_model_config, load_model_config
from scripts.abt_to_model_lightgbm import (
    compute_metrics,
    evaluate_model,
    get_s3_filesystem,
    print_metrics,
    qa_abt_load,
    read_parquet,
    run_training,
    split_abt_three_way,
    split_features_target,
    prepare_boosters_data,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

config = load_model_config(ROOT / 'config' / 'model_config.yaml')
print('Config carregada:', config.config_path)
print(f"Trilho={config.feature_set} | threshold={config.business_threshold} | seed={config.random_state}")
print(
    f"Splits: treino={config.split_train:.1%} | teste={config.split_test:.1%} | demo={config.split_demo:.1%}"
)


## 1. Carga da ABT e split 80 / 19,9 / 0,1

O holdout **demo** é salvo em `Dados/abt/abt_demo_holdout.parquet` e **nunca** entra no treino.


In [ ]:
abt_path = config.resolve_abt_path()
fs = get_s3_filesystem() if abt_path.startswith('s3://') else None
abt = read_parquet(abt_path, fs)
qa_abt_load(abt, config)

print(f'ABT: {abt_path}')
print(f'Linhas: {len(abt):,} | Colunas: {abt.shape[1]}')
print(f'Demo holdout será salvo em: {config.resolve_demo_holdout_path()}')
display(abt['TARGET'].value_counts(normalize=True).mul(100).round(2).to_frame('%'))


## 2. Treino, avaliação e exportação

Executa o pipeline completo (treino no 80%, métricas no 19,9%, exporta `.pkl` + `artifacts/model_metadata.json`).


In [ ]:
model_path = run_training(config=config)
metadata_path = config.resolve_metadata_path(prefer_s3=model_path.startswith('s3://'))
print('Modelo exportado:', model_path)
print('Metadados:', metadata_path)


## 3. Consulta rápida no holdout demo (0,1%)

Exemplo para a avaliação individual: escolher um `SK_ID_CURR` que o modelo nunca viu.


In [ ]:
import pickle
import json

demo = pd.read_parquet(config.resolve_demo_holdout_path())
sample = demo.iloc[0]
print('Cliente demo:', int(sample['SK_ID_CURR']), '| TARGET real:', int(sample['TARGET']))

if Path(model_path).exists():
    with open(model_path, 'rb') as handle:
        model = pickle.load(handle)
else:
    print('Modelo remoto — carregue via scripts/predict.py no ambiente Docker')

X_demo, y_demo = split_features_target(demo.iloc[[0]], config)
cat_cols = X_demo.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
X_demo_boost, _ = prepare_boosters_data(X_demo, X_demo, cat_cols)
proba = model.predict_proba(X_demo_boost)[:, 1][0]
pred = int(proba >= config.business_threshold)
print(f'P(inadimplente)={proba:.4f} | predição(t={config.business_threshold})={pred}')

meta_file = Path(config.resolve_metadata_path())
if meta_file.exists():
    display(json.loads(meta_file.read_text(encoding='utf-8')))
